# Implementation of the equations that make up the GSW functions for calculating the seawater surface density. A summary of the necessary steps are given in the cell below. 

# Mathematical equations

The **main equation** used is: 

$$
\hat{v} (S_A \Theta, p ) = v_u \sum_{i,j,k} v_{ijk} s^i \tau^j \pi^k 
$$

Where the values associated with i,j,k are given in the TEOS10 manual in Appendix K. 

$$
s = \sqrt{\frac{S_A + 24 gkg^-1}{S_{A_u}}}, \quad S_{A_u} = \frac{40 \times 35.16504 g kg^{-1}}{35}
$$

$$
\tau = \frac{\Theta}{\Theta_u}, \quad \Theta_u = 40^{\circ} C  
$$

$$
\pi = \frac{p}{p_u}, \quad P_u = 10^4 dbar
$$

$$
v_u = 1m^{3} kg^{-1}, \quad 
$$

However, p = 0 in this function and therefore $\pi \rightarrow 0$ when $k > 0$. Thats why the GSW function only needs salinity and temperature, not pressure for calculating the surface density: 

$$\hat{v} (S_A, \Theta, p = 0)$$


**Furthermore**, we also need to calculate the conserved temperature to use in the main equation. The following related equations are dedicated to this process.

For calculating the conserved temperature, we need to calculate both the entropy and a potential entalpi! The following steps are therefore needed in the calculations:

1. Calculation of the entropy (equation 2.10.1 from the TEOS manual): 

$$
\eta = \eta(S_A, t, p) = -g_T = - \frac{\partial g_T}{\partial T}
$$

In TEOS - empirical values are already used to create polynomial functions of $\eta$ - and I will rather use these estimated values directly with the aimn to reduce the need for computational calculations. These approximated values are found in appendix B of the TEOS and Copernicus manual! https://os.copernicus.org/articles/19/1719/2023/os-19-1719-2023.pdf

2. An iterativ calculation of potential temperature for the surface - ie. $\theta_0$. This is calculated using the Newton-Raphson iterative technique as illustrated in the equation below, found from TEOS10. It can also be estimated as an integral of the adiabatic lapse rate (Fofonoff, 1962 \& 1985), but I also suspect this would require potentially unneseccary computational power. 

$$
\eta (S_A, \theta, p_r) = \eta (S_A, t, p)
$$

The Newton Raphson iterative process is:
$$
x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}
$$

Where we have to differentiate $g_t$ in our calculations. Furthermore, the full thermodynamic equation of entropy is given as: 

$$
\hat{\eta} (S_A, \Theta) = c_p^0 ln(1 + \Theta / T_0) + \alpha (\frac{S_A}{S_{SO}}) ln (\frac{S_A}{S_{SO}}) + P \{ 8,8 \} (s, \tau)
$$

Where: 

$$
T_0 = 273.15, \quad c_p^0 = 3991.86795711963 , \quad \alpha = - 9.309495003228781 , \quad S_{SO} = 35.16504 
$$

and the polynomial is differentiated such that the potency of $\tau$ \& $s$ is subtracted according to standard differentiation rules. 

3. Then we have to calculate the potential entalpy, where the reference pressure is always set to be zero because most heat flux activity is near the sea-surface. The reference pressure $p_r = 0$ dbar. This is calculated by using: 

$$
h^0 (S_A, t, p) = h (S_A, \theta, 0) = g(S_A, \theta, 0) - (T_0 + \theta)g_T (S_A, \theta, 03)
$$ 

Shortened to (I think - this is my doing hehe):
$$
h^0 = G - T * g_t 
$$

4. And yuhu now we can finally calculate the conserved temperature! Again with the use of the TEOS equations: 

$$
\Theta (S_A, t, p) = \tilde{\Theta} (S_A, \theta) = \frac{\tilde{h^0}(S_A, \theta)}{c_p^0}

$$

In [1]:
def Gibbs(tau, salinity):
    #The Gibbs empirical values gathered from Copernicus - for easier calculations of the Gibbs polynomials
    #The polynomials reach a highest power of 8
    
    """
    Constants
    """
    p_8 = {
    (0,0) : - 3.7102436569e-01,
    (1,0) :  3.0834502223e-04  , 
    (2,0) : - 3.2916987818e+00, 
    (3,0) :  7.2818259040e+00 , 
    (4,0) : - 5.6657256773e+00, 
    (5,0) :  2.8402903938e+00 , 
    (6,0) : - 8.9615123138e-01, 
    (7,0) : 1.0035964794e-01  ,
    (8,0) : 1.8140964105e-03  ,
    (0,1) : 3.0779211774e-02  ,
    (1,1) : 1.5006196848e-03  ,
    (2,1) : 1.2029316021e-01  ,
    (3,1) : 3.7464975805e-01  ,
    (4,1) : - 6.0590428227e-01,
    (5,1) : 6.4365865093e-02  ,
    (6,1) : 2.4626795446e-02  ,
    (7,1) : - 1.0335853091e-02 ,
    (0,2) : 2.3045093877e+00 ,
    (1,2) : - 5.4154968624e-03 ,
    (2,2) : - 2.5098282844e+00,
    (3,2) : 1.9163697628e-02,
    (4,2) : 9.6230320461e-02,
    (5,2) : 3.7953034101e-02,
    (6,2) : - 5.1206778774e-04,

    (0,3) : - 8.4974032876e-01 ,
    (1,3) : - 1.3727475447e-02,
    (2,3) : 8.6969911602e-01,
    (3,3) : 1.1127539375e-01,
    (4,3) : - 8.7616123860e-02,
    (5,3) : - 1.6250024449e-02,
    (0,4) : 4.1807750439e-01,
    (1,4) : 5.1388181100e-02,
    (2,4) : - 3.1917000611e-01,
    (3,4) : - 4.4999965986e-02,
    (4,4) : 3.3822211876e-02,
    (0,5) : - 1.9191736060e-01,
    (1,5) : - 5.3890029514e-02,
    (2,5) : 9.3472917957e-02,
    (3,5) : - 4.9779616704e-04,
    (0,6) : 6.6066546976e-02,
    (1,6) : 2.4144978278e-02,
    (2,6) : -1.2850921670e-02,
    (0,7) : -1.3678360946e-02,
    (1,7) : -4.1337102429e-03,
    (0,8) : 1.1180283076e-03}

    #Then we calculate the polynomial values where the potency is differentiated one time and included in the values
    theta0 = 40.0 #given in article of thermodynamic potential in seawater 
    gibbs_poly_derivative = 0.0
    for (i,j), coeff in p_8.items():
        if j > 0:
            gibbs_poly_derivative += j * coeff * (salinity **i) * (tau**(j-1))
    #Now because we are differentiating based on  T (which is what we have) - we will need to scale our tau to get the non-dimensional right var
    gibbs_poly_derivative_scaled = gibbs_poly_derivative /  theta0

    return gibbs_poly_derivative_scaled

In [8]:
tau = 0.25
salinity = 0.875
result = Gibbs(tau, salinity)
print(result)

%pip install gsw 
import gsw

0.006739317507869232

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [28]:
#scaling vars for testing
theta0 = 40
sso = 35.16504
temp_gsw = tau * theta0
salinity_gsw = salinity ** 2 * sso
dfdf = gsw.entropy_from_CT(salinity_gsw, temp_gsw)
print(dfdf)

145.23672419560265


In [ ]:
def iterative_potential_temp(salinity, temperature):
    """
    Definition: 
    Calculating the iterative potential temperature. Eta varies with absolute salinity and temperature,
    so we differentiate the gibbson poly sum with respect to the temperature. 
    
    """
    theta = temperature
    tolerance = 1e-14 #as stated in potential temperature section 3.1 from IOC et al 2010
    max_iterations = 100 

    for i in range(max_iterations):
        

        



